In [0]:
data = [
    ("U001", "2024-07-16 10:00:00"),
    ("U001", "2024-07-16 10:03:00"), # < 5 min (suspicious)
    ("U001", "2024-07-16 10:20:00"),
    ("U002", "2024-07-16 11:00:00"),
    ("U002", "2024-07-16 11:02:00"), # < 5 min (suspicious)
    ("U002", "2024-07-16 11:10:00"),
    ("U003", "2024-07-16 12:00:00"),
    ("U003", "2024-07-16 12:07:00"), # > 5 min (not suspicious)
]

df = spark.createDataFrame(data, ["user_id", "timestamp"])
display(df)

In [0]:
df = df.withColumn('timestamp', to_timestamp('timestamp'))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

window_spec = Window.partitionBy('user_id').orderBy(desc('timestamp'))

df_res = df.withColumn('lag_date',lag('timestamp').over(window_spec))\
           .withColumn('diff',(unix_timestamp('lag_date')-unix_timestamp('timestamp'))/60)\
           .filter((col('diff').isNotNull()) & (col('diff')<5)) \
           .withColumn('suspicious_flag',col('diff')<5)
df_res.show()